In [ ]:
import pandas as pd
import os
import sys
import numpy as np
import re

In [55]:
species = pd.read_csv("classifications/species.csv")
flyway = pd.read_csv("classifications/flyway.csv")
meta = pd.read_csv("meta/metadata.csv")

In [56]:
meta['region'] = meta['region'].str.lower()
meta['host'] = meta['host'].str.lower()
flyway['location'] = flyway['location'].str.lower()

In [57]:
df_avian = meta.loc[
    (meta['region'] == 'north america') & (meta['host'] == 'avian')
].copy() 

In [58]:
df_avian['updated_species'] = df_avian['strain'].str.lower().str.split("/").str[1]
df_avian = df_avian.dropna(subset=['species'])
df_avian = df_avian.drop(columns=['broad'])

In [59]:
species['annotated'] = species['annotated'].str.lower()

df_avian = df_avian.merge(
    species[['annotated', 'correction', 'order', 'broad']],
    left_on='updated_species',
    right_on='annotated',
    how='left'
).copy()


# # Optional: drop helper columns from merge
# df_avian = df_avian.drop(columns=['annotated', 'correction', 'order_y'])
# df_avian = df_avian.rename(columns={'order_x': 'order'})


In [60]:
df_avian['species'] = df_avian['correction'].str.lower()
df_avian['order'] = df_avian['order_y'].str.lower() 
df_avian = df_avian.drop(columns=['updated_species', 'annotated', 'correction', 'order_y', 'order_x'])
df_avian

,strain,accession,subtype,date,host,country,region,species,broad,order
0,A/American_black_duck/Alberta/284/2016|2016-08-28,MG599652,H3N8,2016-08-28,avian,Canada,north america,american_black_duck,duck,anseriformes
1,A/American_black_duck/Alberta/96/2016|2016-08-11,MF613845,H3N6,2016-08-11,avian,Canada,north america,american_black_duck,duck,anseriformes
2,A/American_black_duck/Maine/44411/532/2008|200...,KP636478,H3N8,2008-01-01,avian,Usa,north america,american_black_duck,duck,anseriformes
3,A/American_Black_Duck/Maine/A00090256/2007|200...,MN210123,H3N2,2007-09-25,avian,Usa,north america,american_black_duck,duck,anseriformes
4,A/American_black_duck/Maine/ME-Y17-170/2017|20...,EPI1774726,H3N8,2017-08-29,avian,Usa,north america,american_black_duck,duck,anseriformes
...,...,...,...,...,...,...,...,...,...,...
2218,A/wood_duck/Ohio/13OS3300/2013|2013-10-12,KJ568401,H3N8,2013-10-12,avian,Usa,north america,wood_duck,duck,anseriformes
2219,A/Wood_Duck/South_Dakota/A00605524/2008|2008-0...,OQ987893,H3N8,2008-08-18,avian,Usa,north america,wood_duck,duck,anseriformes
2220,A/wood_duck/Wisconsin/10OS2778/2010|2010-09-26,CY132957,H3N8,2010-09-26,avian,Usa,north america,wood_duck,duck,anseriformes
2221,A/wood_duck/Wisconsin/11OS2912/2011|2011-09-24,CY167110,H3N6,2011-09-24,avian,Usa,north america,wood_duck,duck,anseriformes


In [61]:
df_avian['state'] = df_avian['strain'].str.lower().str.split("/").str[2]

In [62]:
df_avian['state'].unique()

array(['alberta', 'maine', 'new_brunswick', 'new_hampshire',
       'newfoundland', 'newfoundland_and_labrador', 'north_carolina',
       'nova_scotia', 'ohio', 'prince_edward_island', 'quebec',
       "st._john's", 'washington', 'oregon', '17os4133', 'alaska',
       'idaho', 'illinois', 'interior_alaska', 'mississippi', 'missouri',
       'montana', 'north_dakota', 'texas', 'wisconsin', 'minnesota',
       'utah', 'new_mexico', 'arizona', 'alb', 'canada', 'guatemala',
       'iowa', 'kansas', 'louisiana', 'lousiana', 'manitoba', 'mexico',
       'saskatchewan', 'south_dakota', 'ny', 'california', 'la',
       'memohis', 'memphis', 'ontario', 'southcentral_alaska', 'michigan',
       'nj', 'new_jersey', 'delaware_bay', 'maryland', '708', 'ab',
       'arkansas', 'bc', 'british_columbia', 'mn', 'nevada', 'new_york',
       'pennsylvania', 'vermont', 'delaware', 'barrow', 'qc', 'georgia',
       'south_carolina', 'nunavet', '_minnesota', 'canada-bc',
       'canada-manitoba', 'canada-on

In [63]:
merged = df_avian.merge(
    flyway[['location', 'flyway']],
    left_on='state',
    right_on='location',
    how='left'
)

In [64]:
merged

,strain,accession,subtype,date,host,country,region,species,broad,order,state,location,flyway
0,A/American_black_duck/Alberta/284/2016|2016-08-28,MG599652,H3N8,2016-08-28,avian,Canada,north america,american_black_duck,duck,anseriformes,alberta,alberta,central_flyway
1,A/American_black_duck/Alberta/96/2016|2016-08-11,MF613845,H3N6,2016-08-11,avian,Canada,north america,american_black_duck,duck,anseriformes,alberta,alberta,central_flyway
2,A/American_black_duck/Maine/44411/532/2008|200...,KP636478,H3N8,2008-01-01,avian,Usa,north america,american_black_duck,duck,anseriformes,maine,maine,atlantic_flyway
3,A/American_Black_Duck/Maine/A00090256/2007|200...,MN210123,H3N2,2007-09-25,avian,Usa,north america,american_black_duck,duck,anseriformes,maine,maine,atlantic_flyway
4,A/American_black_duck/Maine/ME-Y17-170/2017|20...,EPI1774726,H3N8,2017-08-29,avian,Usa,north america,american_black_duck,duck,anseriformes,maine,maine,atlantic_flyway
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2231,A/wood_duck/Ohio/13OS3300/2013|2013-10-12,KJ568401,H3N8,2013-10-12,avian,Usa,north america,wood_duck,duck,anseriformes,ohio,ohio,mississippi_flyway
2232,A/Wood_Duck/South_Dakota/A00605524/2008|2008-0...,OQ987893,H3N8,2008-08-18,avian,Usa,north america,wood_duck,duck,anseriformes,south_dakota,south_dakota,central_flyway
2233,A/wood_duck/Wisconsin/10OS2778/2010|2010-09-26,CY132957,H3N8,2010-09-26,avian,Usa,north america,wood_duck,duck,anseriformes,wisconsin,wisconsin,mississippi_flyway
2234,A/wood_duck/Wisconsin/11OS2912/2011|2011-09-24,CY167110,H3N6,2011-09-24,avian,Usa,north america,wood_duck,duck,anseriformes,wisconsin,wisconsin,mississippi_flyway


In [65]:
merged['order'].value_counts()

order
anseriformes       1972
charadriiformes     173
avian                50
galliformes          39
neoave                2
Name: count, dtype: int64

In [66]:
merged['broad'].value_counts()

broad
duck           1950
wader           162
avian            50
turkey           36
goose            22
gull             11
coot              1
chicken           1
cormorant         1
guinea_fowl       1
quail             1
Name: count, dtype: int64

In [67]:
merged = merged.drop_duplicates(subset=["strain"])
merged.to_csv("meta/meta.csv", index=False)